# Environment Verification

This notebook validates that the DataGovKG development environment is correctly set up.

## Components tested

| Component | Purpose |
|-----------|---------|
| Python 3.11 | Programming language |
| `rdflib` | RDF graph manipulation |
| `SPARQLWrapper` | Communication with Fuseki via HTTP |
| `pyshacl` | SHACL validation engine |
| `openai` | LLM API client (for AI agents) |
| `streamlit` | Dashboard UI framework |
| Apache Jena Fuseki | Triplestore (running in Docker on port 3030) |

## Success criteria

- All Python libraries import without errors
- Fuseki responds to HTTP requests at `http://localhost:3030`
- A test RDF triple can be inserted and retrieved via SPARQL

If all cells run without errors, the environment is operational.

In [3]:
# ─────────────────────────────────────────────────────────────
#  Python Environment Check
# Purpose: Verify Python version and confirm we're running 
#          inside the 'datagovkg' conda environment.
# ─────────────────────────────────────────────────────────────

import sys      # built-in module for accessing system info
import platform # built-in module for OS/platform details

# Print a clear header (the '=' multiplied 60 times = a visual divider)
print("=" * 60)
print("PYTHON ENVIRONMENT CHECK")
print("=" * 60)

# Display Python version (major.minor.patch)
# Example output: "Python 3.11.15"
print(f"Python version  : {platform.python_version()}")

# Display Python executable path
# This tells us WHICH Python is running (very important for debugging)
print(f"Python location : {sys.executable}")

# Display platform info (Windows version, architecture)
print(f"Platform        : {platform.platform()}")

# Verify we're in the correct conda environment
# The path should contain 'envs\datagovkg' or 'envs/datagovkg'
if "datagovkg" in sys.executable:
    print("\n Running in 'datagovkg' environment")
else:
    print("\n  WARNING: Not in 'datagovkg' environment!")
    print("    Restart Jupyter from within the datagovkg conda environment.")

PYTHON ENVIRONMENT CHECK
Python version  : 3.11.15
Python location : C:\Users\bsula\anaconda3\envs\datagovkg\python.exe
Platform        : Windows-10-10.0.26200-SP0

 Running in 'datagovkg' environment


In [5]:
# Core Libraries Version Check
# Uses importlib.metadata for a universal way to get versions
# (works for all libraries, including older ones like owlready2)

from importlib.metadata import version, PackageNotFoundError

# List of libraries we need to verify
required_libraries = [
    # Semantic Web stack
    "rdflib",
    "pyshacl",
    "SPARQLWrapper",
    "owlready2",
    # AI & LLM
    "openai",
    # User Interface
    "streamlit",
    # Data Processing
    "pandas",
]

# Display header
print("=" * 60)
print("INSTALLED LIBRARIES")
print("=" * 60)
print(f"{'Library':<20} {'Version':<15} {'Status'}")
print("-" * 60)

# Check each library
verified_count = 0
for lib_name in required_libraries:
    try:
        # version() returns the installed version as a string
        lib_version = version(lib_name)
        status = "OK"
        verified_count += 1
    except PackageNotFoundError:
        # Library is not installed
        lib_version = "N/A"
        status = "MISSING"
    
    # Print row with aligned columns
    print(f"{lib_name:<20} {lib_version:<15} {status}")

print("-" * 60)
print(f"Total libraries verified: {verified_count} / {len(required_libraries)}")

INSTALLED LIBRARIES
Library              Version         Status
------------------------------------------------------------
rdflib               7.1.4           OK
pyshacl              0.30.1          OK
SPARQLWrapper        2.0.0           OK
owlready2            0.48            OK
openai               1.59.7          OK
streamlit            1.41.1          OK
pandas               2.2.3           OK
------------------------------------------------------------
Total libraries verified: 7 / 7


In [8]:
# Fuseki Connectivity Test
# Purpose: Verify Fuseki is running and dataset is accessible.
# Uses admin credentials for management endpoints.

import requests

# Define Fuseki server URLs
FUSEKI_BASE     = "http://localhost:3030"
FUSEKI_DATASET  = f"{FUSEKI_BASE}/datagovkg"
FUSEKI_QUERY    = f"{FUSEKI_DATASET}/sparql"

# Admin credentials (development only - not for production)
# Fuseki requires authentication for management endpoints
FUSEKI_AUTH = ("admin", "admin")

print("=" * 60)
print("FUSEKI CONNECTIVITY TEST")
print("=" * 60)
print(f"Server URL : {FUSEKI_BASE}")
print(f"Dataset    : {FUSEKI_DATASET}")
print(f"SPARQL URL : {FUSEKI_QUERY}")
print("-" * 60)

all_tests_passed = True

# ─── Test 1: Server reachability ───────────────────────────
print("\n[Test 1/3] Server reachability...")
try:
    # Ping endpoint is public - no auth needed
    response = requests.get(f"{FUSEKI_BASE}/$/ping", timeout=5)
    
    if response.status_code == 200:
        print(f"          Server responded with status {response.status_code} - OK")
    else:
        print(f"          Server responded with status {response.status_code} - FAIL")
        all_tests_passed = False
except requests.exceptions.ConnectionError:
    print("          Cannot connect to server - FAIL")
    print("          Check that Fuseki Docker container is running: docker ps")
    all_tests_passed = False

# ─── Test 2: Dataset exists (requires admin auth) ─────────
print("\n[Test 2/3] Dataset 'datagovkg' exists...")
try:
    # Management endpoint requires authentication
    # We pass auth=(username, password) using HTTP Basic Auth
    response = requests.get(
        f"{FUSEKI_BASE}/$/datasets",
        auth=FUSEKI_AUTH,
        timeout=5
    )
    
    if response.status_code == 200:
        datasets_info = response.json()
        dataset_names = [ds["ds.name"] for ds in datasets_info.get("datasets", [])]
        
        if "/datagovkg" in dataset_names:
            print(f"          Dataset '/datagovkg' found - OK")
        else:
            print(f"          Dataset '/datagovkg' NOT found - FAIL")
            print(f"          Available datasets: {dataset_names}")
            all_tests_passed = False
    else:
        print(f"          Failed (status {response.status_code}) - FAIL")
        all_tests_passed = False
except Exception as e:
    print(f"          Error: {e} - FAIL")
    all_tests_passed = False

# ─── Test 3: SPARQL endpoint (public - no auth) ───────────
print("\n[Test 3/3] SPARQL endpoint accepts queries...")
try:
    sparql_query = "SELECT (COUNT(*) AS ?count) WHERE { ?s ?p ?o }"
    
    response = requests.post(
        FUSEKI_QUERY,
        data={"query": sparql_query},
        headers={"Accept": "application/sparql-results+json"},
        timeout=5
    )
    
    if response.status_code == 200:
        result = response.json()
        triple_count = result["results"]["bindings"][0]["count"]["value"]
        print(f"          Query succeeded - {triple_count} triples in dataset - OK")
    else:
        print(f"          Query failed (status {response.status_code}) - FAIL")
        all_tests_passed = False
except Exception as e:
    print(f"          Error: {e} - FAIL")
    all_tests_passed = False

# ─── Final Summary ─────────────────────────────────────────
print("\n" + "=" * 60)
if all_tests_passed:
    print("RESULT: All connectivity tests passed - Fuseki ready to use")
else:
    print("RESULT: Some tests failed - troubleshoot before continuing")
print("=" * 60)

FUSEKI CONNECTIVITY TEST
Server URL : http://localhost:3030
Dataset    : http://localhost:3030/datagovkg
SPARQL URL : http://localhost:3030/datagovkg/sparql
------------------------------------------------------------

[Test 1/3] Server reachability...
          Server responded with status 200 - OK

[Test 2/3] Dataset 'datagovkg' exists...
          Dataset '/datagovkg' found - OK

[Test 3/3] SPARQL endpoint accepts queries...
          Query succeeded - 0 triples in dataset - OK

RESULT: All connectivity tests passed - Fuseki ready to use


In [9]:
# End-to-End RDF Pipeline Test
# Purpose: Demonstrate the complete workflow we'll use throughout 
#          the project: create -> insert -> retrieve -> verify.

import requests

# Reuse configuration from previous cell
FUSEKI_QUERY  = "http://localhost:3030/datagovkg/sparql"
FUSEKI_UPDATE = "http://localhost:3030/datagovkg/update"
FUSEKI_AUTH   = ("admin", "admin")

print("=" * 60)
print("END-TO-END RDF PIPELINE TEST")
print("=" * 60)

# ─── Step 1: Define test data ──────────────────────────────
# We use a placeholder URI for the test entity
# The 'urn:' scheme is for temporary/test data (not for production ontology)
test_subject   = "<urn:test:datagovkg-verification>"
test_predicate = "<urn:test:hasStatus>"
test_object    = '"operational"'  # string literal - wrapped in quotes

print("\nTest triple to insert:")
print(f"  Subject   : {test_subject}")
print(f"  Predicate : {test_predicate}")
print(f"  Object    : {test_object}")

# ─── Step 2: Insert the triple via SPARQL UPDATE ──────────
print("\n[Step 1/3] Inserting triple via SPARQL UPDATE...")

# Build the SPARQL UPDATE query
# 'INSERT DATA' adds triples to the default graph
insert_query = f"""
INSERT DATA {{
    {test_subject} {test_predicate} {test_object} .
}}
"""

try:
    # POST the query to the /update endpoint (requires auth)
    response = requests.post(
        FUSEKI_UPDATE,
        data={"update": insert_query},
        auth=FUSEKI_AUTH,
        timeout=5
    )
    
    if response.status_code in [200, 204]:
        # 200 = OK, 204 = No Content (both mean success for UPDATE)
        print(f"          Insert succeeded (status {response.status_code}) - OK")
    else:
        print(f"          Insert failed (status {response.status_code}) - FAIL")
        print(f"          Response: {response.text[:200]}")
except Exception as e:
    print(f"          Error: {e} - FAIL")

# ─── Step 3: Retrieve the triple via SPARQL SELECT ────────
print("\n[Step 2/3] Retrieving triple via SPARQL SELECT...")

# Build a SELECT query that looks for our specific subject
# The '?o' is a variable that captures whatever the object is
select_query = f"""
SELECT ?p ?o WHERE {{
    {test_subject} ?p ?o .
}}
"""

try:
    response = requests.post(
        FUSEKI_QUERY,
        data={"query": select_query},
        headers={"Accept": "application/sparql-results+json"},
        timeout=5
    )
    
    if response.status_code == 200:
        result = response.json()
        bindings = result["results"]["bindings"]
        
        if len(bindings) > 0:
            print(f"          Query succeeded - found {len(bindings)} triple(s) - OK")
            # Display what we retrieved
            for binding in bindings:
                retrieved_p = binding["p"]["value"]
                retrieved_o = binding["o"]["value"]
                print(f"          Retrieved: <{retrieved_p}> '{retrieved_o}'")
        else:
            print(f"          No triples found - FAIL")
    else:
        print(f"          Query failed (status {response.status_code}) - FAIL")
except Exception as e:
    print(f"          Error: {e} - FAIL")

# ─── Step 4: Verify the data matches what we inserted ─────
print("\n[Step 3/3] Verifying data integrity...")

if len(bindings) > 0:
    retrieved_value = bindings[0]["o"]["value"]
    expected_value  = "operational"
    
    if retrieved_value == expected_value:
        print(f"          Retrieved value matches inserted value - OK")
    else:
        print(f"          Mismatch: expected '{expected_value}', got '{retrieved_value}' - FAIL")
else:
    print(f"          Cannot verify - no data retrieved - FAIL")

# ─── Final Summary ─────────────────────────────────────────
print("\n" + "=" * 60)
print("PIPELINE TEST COMPLETE")
print("=" * 60)
print("The RDF roundtrip (Python -> Fuseki -> Python) is working.")
print("Ready to start building the ontology family.")

END-TO-END RDF PIPELINE TEST

Test triple to insert:
  Subject   : <urn:test:datagovkg-verification>
  Predicate : <urn:test:hasStatus>
  Object    : "operational"

[Step 1/3] Inserting triple via SPARQL UPDATE...
          Insert succeeded (status 200) - OK

[Step 2/3] Retrieving triple via SPARQL SELECT...
          Query succeeded - found 1 triple(s) - OK
          Retrieved: <urn:test:hasStatus> 'operational'

[Step 3/3] Verifying data integrity...
          Retrieved value matches inserted value - OK

PIPELINE TEST COMPLETE
The RDF roundtrip (Python -> Fuseki -> Python) is working.
Ready to start building the ontology family.


In [10]:
# Display All Triples in Fuseki Dataset
# A utility cell that can be re-run anytime to inspect the dataset.
# Useful for verifying inserts, debugging, and confirming data integrity.

import requests

FUSEKI_QUERY = "http://localhost:3030/datagovkg/sparql"

# SPARQL query to retrieve ALL triples in the dataset
# The pattern '?s ?p ?o' matches any subject, predicate, object
all_triples_query = """
SELECT ?subject ?predicate ?object
WHERE {
    ?subject ?predicate ?object .
}
ORDER BY ?subject ?predicate
"""

# Execute the query
response = requests.post(
    FUSEKI_QUERY,
    data={"query": all_triples_query},
    headers={"Accept": "application/sparql-results+json"},
    timeout=5
)

# Parse the JSON response
result = response.json()
bindings = result["results"]["bindings"]

# Display header
print("=" * 75)
print(f"FUSEKI DATASET CONTENTS - {len(bindings)} triple(s) found")
print("=" * 75)

if len(bindings) == 0:
    print("\n(Dataset is empty)")
else:
    # Display each triple with clear labels and visual separation
    for i, binding in enumerate(bindings, start=1):
        # Extract values from the JSON response structure
        subject   = binding["subject"]["value"]
        predicate = binding["predicate"]["value"]
        obj       = binding["object"]["value"]
        
        # Get the data type of the object (URI, literal, etc.)
        obj_type = binding["object"]["type"]
        
        print(f"\nTriple #{i}")
        print(f"  Subject   : <{subject}>")
        print(f"  Predicate : <{predicate}>")
        # Format object based on its type
        if obj_type == "literal":
            print(f"  Object    : \"{obj}\" (literal)")
        elif obj_type == "uri":
            print(f"  Object    : <{obj}> (URI)")
        else:
            print(f"  Object    : {obj} ({obj_type})")

print("\n" + "=" * 75)
print("End of dataset contents")
print("=" * 75)

FUSEKI DATASET CONTENTS - 1 triple(s) found

Triple #1
  Subject   : <urn:test:datagovkg-verification>
  Predicate : <urn:test:hasStatus>
  Object    : "operational" (literal)

End of dataset contents
